In [ ]:
import torch
import torch.nn as nn

from tqdm import tqdm
from pathlib import Path

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
DATA_DIR: Path = Path("data_full/dataset")
RESULTS_DIR: Path = Path("results/heatmaps")

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2 as transforms_v2
from torchvision import transforms
from torchvision.datasets import ImageFolder

In [ ]:
BATCH_SIZE:  int = 10
NUM_WORKERS: int = 2

IMAGENET1K_MEAN = [0.485, 0.456, 0.406]
IMAGENET1K_STD  = [0.229, 0.224, 0.225]

In [ ]:
test_transforms = transforms_v2.Compose(
    [
        transforms_v2.Resize(size=(224 + 8, 224 + 8), antialias=True),
        transforms_v2.CenterCrop(224),
        transforms_v2.ToImage(),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=IMAGENET1K_MEAN, std=IMAGENET1K_STD),
    ]
)


test_dataset = ImageFolder(
    DATA_DIR/"test", 
    transform=test_transforms,
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [ ]:
class CosineSimilarityCodebook(nn.Module):
    def __init__(self, num_entries: int, embedding_dim: int):
        super().__init__()
        self.P =  num_entries
        self.embeddings = nn.Embedding(num_entries, embedding_dim)

        with torch.no_grad():
            self.embeddings.weight.copy_(torch.randn(num_entries, embedding_dim) * (embedding_dim ** -0.5))

        self.return_idxs = False

        self.out_map = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim),
        )
        self.out_norm = nn.LayerNorm(embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert len(x.shape) == 4, "Input tensor must have a batch dimension"

        B, C, H, W = x.shape
        x = x.view(B, C, H*W).permute(0, 2, 1)

        x = torch.functional.F.normalize(x, dim=-1)
        c = torch.functional.F.normalize(
            self.embeddings.weight, 
            dim=-1,
        )

        sim = torch.matmul(x, c.t())
        indices = torch.argmax(sim, dim=-1)
        sim_val, _ = torch.max(sim, dim=-1)

        q = self.embeddings(indices)

        q = self.out_norm(q + self.out_map(q))
        q = q.view(B, H, W, C).permute(0, 3, 1, 2)
        
        if self.return_idxs:
            indices = indices.view(B, H, W)
            sim_val = sim_val.view(B, H, W)
            sim = sim.view(B, H, W, self.P)
            return q, indices, sim_val, sim
        return q

In [ ]:
NUM_CLASSES: int = 200
EMBED_DIM: int = 768
NUM_PROTOS: int = 512

In [ ]:
from torchvision import models

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

model.classifier.pop(-1)
model.classifier.append(nn.Linear(EMBED_DIM, NUM_CLASSES))

codebook = CosineSimilarityCodebook(num_entries=NUM_PROTOS, embedding_dim=EMBED_DIM)
model.features.add_module("codebook", codebook)

model.load_state_dict(torch.load(
    f"codebook_2025-02-19_21-28-54_v2.pth",
    # map_location=torch.device('cpu'),
))
_ = model.eval()

In [ ]:
model.features.codebook.return_idxs = True

In [ ]:
import os 

for idx in range(NUM_PROTOS):
    os.makedirs(RESULTS_DIR/str(idx), exist_ok=True)

In [ ]:
import heapq
import torch
from tqdm import tqdm

top_k = {idx: [] for idx in range(NUM_PROTOS)}
k = 10

for batch_idx, (x, _) in tqdm(enumerate(test_loader), total=len(test_loader)):
    _, _, _, z = model.features(x)

    batch_size, fmap_h, fmap_w, num_protos = z.shape
    z = z.view(batch_size, -1, num_protos)
    max_act, _ = torch.max(z, dim=1, keepdims=False)

    for b in range(batch_size):
        image_idx = batch_idx * batch_size + b

        for p in range(num_protos):
            act = max_act[b, p].item()

            heapq.heappush(top_k[p], (act, image_idx))

In [ ]:
for idx in top_k:
    top_k[idx].sort(reverse=True, key=lambda x: x[0])
    top_k[idx] = top_k[idx][:k]

In [ ]:
top_k[0]

In [ ]:
test_transforms = transforms_v2.Compose(
    [
        transforms_v2.Resize(size=(224 + 8, 224 + 8), antialias=True),
        transforms_v2.CenterCrop(224),
        transforms_v2.ToImage(),
        transforms_v2.ToDtype(torch.float32, scale=True),
    ]
)


test_dataset = ImageFolder(
    DATA_DIR/"test", 
    transform=test_transforms,
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

In [ ]:
normalize = transforms_v2.Normalize(
    mean=IMAGENET1K_MEAN, 
    std=IMAGENET1K_STD
)

In [ ]:
import numpy as np
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

to_pil = transforms.ToPILImage()


for proto_idx in tqdm(top_k):
    proto_top_k = top_k[proto_idx]
    
    for smp_idx, (act, image_idx) in enumerate(proto_top_k):
    
        img, _ = test_loader.dataset[image_idx]
        
        input = normalize(img.unsqueeze(0))
        _, _, _, z = model.features(input)
        fmap = z[0, ..., proto_idx].detach().cpu().numpy()
        
        img = img.permute(1, 2, 0).cpu().numpy()
        fmap = zoom(fmap, (224 / 7, 224 / 7), order=3)
    
        plt.figure()
        plt.subplot(121)
        plt.imshow(img)
        plt.axis(False)
        
        plt.subplot(122)
        plt.title(f"act: {str(round(act, 3))}")
        plt.imshow(img)
        plt.imshow(fmap, alpha=0.6, cmap="jet")
        plt.colorbar()
        plt.axis(False)
    
        plt.savefig(RESULTS_DIR/str(proto_idx)/f"{smp_idx}.jpg")
        plt.close()
    